# 多渠道适配器（Channel Adapter）

## 为什么需要渠道适配器？

B站商业化 Agent 需要同时接入多个平台渠道：

| 渠道 | 场景 | 消息格式 |
|------|------|----------|
| **Web/H5** | B站官网广告后台 | HTTP JSON |
| **API** | 第三方广告主系统集成 | RESTful / gRPC |
| **微信** | 客服/UP主沟通 | XML（微信公众平台） |
| **飞书** | 内部运营/审核工作流 | JSON（飞书开放平台） |
| **钉钉** | 部分合作方对接 | JSON（钉钉机器人） |

不同渠道的消息格式、认证方式、回调机制各不相同。**适配器模式（Adapter Pattern）** 将这些差异封装在适配器内部，让核心 Agent 逻辑与渠道无关。

### 架构示意
```
微信消息 ──→ WeChatAdapter ──┐
飞书消息 ──→ FeishuAdapter  ──┤
Web请求  ──→ WebAdapter     ──┼──→ Gateway ──→ Agent ──→ Runner
API调用  ──→ ApiAdapter     ──┤
钉钉消息 ──→ DingAdapter    ──┘
```

适配器负责：**协议转换**（入站）+ **格式适配**（出站），核心 Agent 只处理统一的内部消息格式。

In [ ]:
# 渠道适配器接口与飞书适配器实现

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any
import json
import time

# ---- 内部统一消息格式 ----
@dataclass
class UnifiedMessage:
    """OpenClaw 内部统一消息格式，所有渠道消息都转换为此格式"""
    channel: str            # 来源渠道
    user_id: str            # 用户标识
    content: str            # 消息内容
    msg_type: str = "text"  # text / image / card
    metadata: dict = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)

@dataclass
class UnifiedResponse:
    """统一响应格式"""
    content: str
    msg_type: str = "text"
    extra: dict = field(default_factory=dict)

# ---- 适配器抽象基类 ----
class ChannelAdapter(ABC):
    """渠道适配器接口"""
    
    @property
    @abstractmethod
    def channel_name(self) -> str:
        ...
    
    @abstractmethod
    def parse_inbound(self, raw_data: dict) -> UnifiedMessage:
        """将渠道原始消息转换为统一格式（入站适配）"""
        ...
    
    @abstractmethod
    def format_outbound(self, response: UnifiedResponse) -> dict:
        """将统一响应转换为渠道格式（出站适配）"""
        ...
    
    @abstractmethod
    def verify_signature(self, raw_data: dict) -> bool:
        """验签（各渠道签名机制不同）"""
        ...

# ---- 飞书适配器实现 ----
class FeishuAdapter(ChannelAdapter):
    """飞书渠道适配器 - B站内部运营沟通"""
    
    def __init__(self, app_id: str, app_secret: str):
        self.app_id = app_id
        self.app_secret = app_secret
    
    @property
    def channel_name(self) -> str:
        return "feishu"
    
    def verify_signature(self, raw_data: dict) -> bool:
        """飞书使用 token 验签"""
        token = raw_data.get("header", {}).get("token", "")
        is_valid = len(token) > 0  # 简化模拟
        print(f"  [飞书验签] token={token[:8]}... → {'通过' if is_valid else '失败'}")
        return is_valid
    
    def parse_inbound(self, raw_data: dict) -> UnifiedMessage:
        """将飞书消息格式转换为统一格式"""
        event = raw_data.get("event", {})
        message = event.get("message", {})
        sender = event.get("sender", {})
        
        # 飞书消息 content 是 JSON 字符串
        content_json = json.loads(message.get("content", '{"text": ""}'))
        
        return UnifiedMessage(
            channel=self.channel_name,
            user_id=sender.get("sender_id", {}).get("open_id", "unknown"),
            content=content_json.get("text", ""),
            msg_type=message.get("message_type", "text"),
            metadata={
                "chat_id": message.get("chat_id", ""),
                "message_id": message.get("message_id", ""),
            }
        )
    
    def format_outbound(self, response: UnifiedResponse) -> dict:
        """将统一响应转换为飞书消息卡片格式"""
        if response.msg_type == "card":
            return {
                "msg_type": "interactive",
                "card": {
                    "header": {"title": {"tag": "plain_text", "content": "Agent 回复"}},
                    "elements": [{"tag": "markdown", "content": response.content}]
                }
            }
        return {
            "msg_type": "text",
            "content": json.dumps({"text": response.content}, ensure_ascii=False)
        }

# ---- Web 适配器实现 ----
class WebAdapter(ChannelAdapter):
    """Web/H5 渠道适配器 - B站广告后台"""
    
    @property
    def channel_name(self) -> str:
        return "web"
    
    def verify_signature(self, raw_data: dict) -> bool:
        return "auth_token" in raw_data
    
    def parse_inbound(self, raw_data: dict) -> UnifiedMessage:
        return UnifiedMessage(
            channel=self.channel_name,
            user_id=raw_data.get("user_id", "unknown"),
            content=raw_data.get("message", ""),
        )
    
    def format_outbound(self, response: UnifiedResponse) -> dict:
        return {"code": 0, "message": "success", "data": {"reply": response.content}}


# ======== 演示 ========
print("=" * 50)
print("飞书渠道适配演示（B站运营场景）")
print("=" * 50)

# 模拟飞书推送的原始消息
feishu_raw = {
    "header": {"token": "verify_token_abc123", "event_type": "im.message.receive_v1"},
    "event": {
        "sender": {"sender_id": {"open_id": "ou_operator_001"}},
        "message": {
            "message_id": "msg_001",
            "chat_id": "oc_chat_001",
            "message_type": "text",
            "content": json.dumps({"text": "查询广告主A的本周投放ROI"})
        }
    }
}

adapter = FeishuAdapter(app_id="cli_xxx", app_secret="secret_xxx")

# 入站适配
print("\n1. 入站适配（飞书格式 → 统一格式）:")
adapter.verify_signature(feishu_raw)
unified_msg = adapter.parse_inbound(feishu_raw)
print(f"  渠道: {unified_msg.channel}")
print(f"  用户: {unified_msg.user_id}")
print(f"  内容: {unified_msg.content}")

# 出站适配
print("\n2. 出站适配（统一格式 → 飞书格式）:")
response = UnifiedResponse(content="广告主A本周ROI为2.8，环比上涨15%", msg_type="card")
feishu_reply = adapter.format_outbound(response)
print(f"  飞书消息类型: {feishu_reply['msg_type']}")
print(f"  回复内容: {json.dumps(feishu_reply, indent=2, ensure_ascii=False)}")

## 适配器模式解析

### 设计模式：Adapter Pattern

```
ChannelAdapter (抽象接口)
  ├── channel_name         # 渠道标识
  ├── verify_signature()   # 验签（各渠道不同）
  ├── parse_inbound()      # 入站：外部格式 → UnifiedMessage
  └── format_outbound()    # 出站：UnifiedResponse → 外部格式

具体实现：
  ├── FeishuAdapter        # 飞书：JSON + 消息卡片
  ├── WebAdapter           # Web：标准 HTTP JSON
  ├── WeChatAdapter        # 微信：XML 格式
  └── DingAdapter          # 钉钉：Webhook JSON
```

### 核心价值

| 优势 | 说明 |
|------|------|
| **解耦** | Agent 核心逻辑与渠道完全解耦，新增渠道零侵入 |
| **统一** | 所有渠道消息转为 UnifiedMessage，下游无感知 |
| **可测** | 适配器可独立单元测试，mock 外部渠道 |
| **可扩展** | 新增渠道只需实现 ChannelAdapter 接口 |

### B站实际应用

- **飞书**：内部运营通过飞书机器人查询广告数据、审核商单
- **Web**：广告主在B站商业化后台与 Agent 交互
- **API**：大广告主通过 API 批量管理投放计划
- **微信**：UP主通过微信接收商单通知和沟通

## 面试速记

### Q1: 为什么用适配器模式而不是在 Gateway 里写 if-else？

> if-else 违反**开闭原则（OCP）**：每新增一个渠道都要修改 Gateway 核心代码。适配器模式让新增渠道变成"增加一个类"，核心代码零改动。B站商业化渠道多且持续新增，适配器模式大幅降低了维护成本。

### Q2: UnifiedMessage 设计的关键字段有哪些？

> - `channel`：来源渠道标识，用于出站时选择对应适配器
> - `user_id`：统一用户标识，跨渠道关联用户
> - `content`：消息内容，统一为纯文本
> - `metadata`：渠道特有信息（如飞书的 chat_id），用于回复时路由

### Q3: 多渠道场景下如何保证消息不丢失？

> 三层保障：（1）适配器层的验签确保消息真实性；（2）消息入站后写入消息队列（如 Kafka），异步处理；（3）出站失败时自动重试，配合死信队列兜底。

### 记忆口诀
```
适配器 = 翻译官：外部格式 ↔ 内部统一格式
新增渠道 = 新增适配器类，核心代码零改动
入站: parse_inbound  出站: format_outbound
```